# PaveScan AI — Crack Detection Model Training

**Train a YOLOv8 segmentation model to detect pavement cracks.**

This notebook is designed to run in **Google Colab with a GPU runtime**.

## How to use:
1. Open this notebook in Google Colab
2. Go to **Runtime → Change runtime type → T4 GPU**
3. Run all cells in order
4. Download the trained model weights at the end
5. Copy `best.pt` into your `pavescan-ai/models/` folder

---

## Two Training Options:

| Option | Dataset | Classes | Images | Type | Best For |
|--------|---------|---------|--------|------|----------|
| **A (Quick Start)** | Crack-Seg | 1 (crack) | 4,029 | Segmentation | Fast results, pixel-level masks |
| **B (Advanced)** | RDD2022 | 4 (crack types + pothole) | 47,420 | Detection | Full inspection system |

**Recommendation:** Start with Option A to get a working model fast, then upgrade to Option B later.

## 1. Setup

In [ ]:
# Install ultralytics (includes YOLOv8)
!pip install -q ultralytics

import torch
from ultralytics import YOLO

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected! Training will be very slow.")
    print("Go to Runtime > Change runtime type > T4 GPU")

---

## Option A: Crack-Seg Dataset (Quick Start)

The Ultralytics Crack-Seg dataset has **4,029 images** with pixel-level segmentation masks. It auto-downloads when you start training — no manual setup needed.

- **1 class:** crack
- **Size:** ~92 MB
- **Training time:** ~15-30 min on T4 GPU

In [ ]:
# Load the pretrained YOLOv8n-seg model (nano = fastest to train)
model = YOLO("yolov8n-seg.pt")

# Train on the crack-seg dataset
# The dataset auto-downloads on first run (~92 MB)
results = model.train(
    data="crack-seg.yaml",   # built-in dataset config
    epochs=50,               # 50 epochs is a good starting point
    imgsz=640,               # standard YOLO input size
    batch=16,                # adjust down to 8 if you hit OOM errors
    name="pavescan_crack",   # name for this training run
    project="runs/segment",  # output directory
    patience=10,             # stop early if no improvement for 10 epochs
    save=True,               # save checkpoints
    plots=True,              # generate training plots
)

### Evaluate the trained model

In [ ]:
# Load the best weights from training
best_model = YOLO("runs/segment/pavescan_crack/weights/best.pt")

# Run validation
metrics = best_model.val()

print("\n=== Model Performance ===")
print(f"Segmentation mAP50:    {metrics.seg.map50:.3f}")
print(f"Segmentation mAP50-95: {metrics.seg.map:.3f}")
print(f"Box mAP50:             {metrics.box.map50:.3f}")
print(f"Box mAP50-95:          {metrics.box.map:.3f}")

### Visualize results

In [ ]:
from IPython.display import Image, display
from pathlib import Path

run_dir = Path("runs/segment/pavescan_crack")

# Show training curves
if (run_dir / "results.png").exists():
    print("Training curves:")
    display(Image(filename=str(run_dir / "results.png"), width=800))

# Show sample predictions on validation set
if (run_dir / "val_batch0_pred.jpg").exists():
    print("\nSample predictions:")
    display(Image(filename=str(run_dir / "val_batch0_pred.jpg"), width=800))

# Show confusion matrix
if (run_dir / "confusion_matrix.png").exists():
    print("\nConfusion matrix:")
    display(Image(filename=str(run_dir / "confusion_matrix.png"), width=600))

### Test on a single image

In [ ]:
import glob
import matplotlib.pyplot as plt
import cv2

# Grab a test image from the dataset
test_images = glob.glob("datasets/crack-seg/images/test/*.jpg")
if test_images:
    test_img = test_images[0]
    results = best_model.predict(test_img, conf=0.25)

    # Show original vs annotated
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    original = cv2.imread(test_img)
    original = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)
    ax1.imshow(original)
    ax1.set_title("Original")
    ax1.axis("off")

    annotated = results[0].plot()
    annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    ax2.imshow(annotated)
    ax2.set_title(f"Detected: {len(results[0].boxes)} cracks")
    ax2.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("No test images found — check dataset path")

### Download trained weights

In [ ]:
from google.colab import files
import shutil

# Copy best weights to a friendly name
best_weights = "runs/segment/pavescan_crack/weights/best.pt"
output_name = "pavescan_crack_seg.pt"
shutil.copy(best_weights, output_name)

print(f"Downloading {output_name}...")
print("After download, copy this file to: pavescan-ai/models/pavescan_crack_seg.pt")
files.download(output_name)

---

## Option B: RDD2022 Dataset (Advanced)

The **Road Damage Dataset 2022** has **47,420 images** from 6 countries with 4 damage types. This gives your system multiple crack categories — much more impressive for a portfolio.

- **4 classes:** Longitudinal Crack, Transverse Crack, Alligator Crack, Pothole
- **Size:** ~3.7 GB (Japan subset: ~520 MB for faster testing)
- **Format:** PascalVOC → we convert to YOLO format below
- **Training time:** ~1-2 hours on T4 GPU (Japan subset)

> **Note:** This is a detection model (bounding boxes), not segmentation. Your PaveScan AI code handles both.

### B1: Download and convert RDD2022

In [ ]:
import os
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path
import shutil
import random

# --- Configuration ---
# Use Japan subset for faster training. Set to "all" to download everything.
COUNTRY = "Japan"  # Options: Japan, India, Czech, Norway, United_States, China, or "all"

# Class mapping: RDD2022 labels → YOLO class IDs
CLASS_MAP = {
    "D00": 0,  # Longitudinal Crack
    "D10": 1,  # Transverse Crack
    "D20": 2,  # Alligator Crack
    "D40": 3,  # Pothole
}
CLASS_NAMES = ["Longitudinal_Crack", "Transverse_Crack", "Alligator_Crack", "Pothole"]

# --- Download ---
print(f"Downloading RDD2022 ({COUNTRY})...")
!wget -q "https://bigdatacup.s3.ap-northeast-1.amazonaws.com/2022/CRDDC2022/RDD2022/Country_Specific_Data_CRDDC2022/{COUNTRY}.zip" -O rdd2022.zip
print("Extracting...")
with zipfile.ZipFile("rdd2022.zip", "r") as z:
    z.extractall("rdd2022_raw")
os.remove("rdd2022.zip")
print("Done!")

In [ ]:
def convert_voc_to_yolo(xml_path, img_width, img_height):
    """Convert a PascalVOC XML annotation to YOLO format lines."""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    lines = []

    for obj in root.findall("object"):
        label = obj.find("name").text
        if label not in CLASS_MAP:
            continue

        class_id = CLASS_MAP[label]
        bbox = obj.find("bndbox")
        xmin = float(bbox.find("xmin").text)
        ymin = float(bbox.find("ymin").text)
        xmax = float(bbox.find("xmax").text)
        ymax = float(bbox.find("ymax").text)

        # Convert to YOLO format: center_x, center_y, width, height (normalized)
        cx = (xmin + xmax) / 2.0 / img_width
        cy = (ymin + ymax) / 2.0 / img_height
        w = (xmax - xmin) / img_width
        h = (ymax - ymin) / img_height

        lines.append(f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

    return lines


# --- Convert dataset ---
from PIL import Image as PILImage

raw_dir = Path("rdd2022_raw")
out_dir = Path("datasets/rdd2022")

# Find the train images and annotations
img_dirs = list(raw_dir.rglob("train/images"))
xml_dirs = list(raw_dir.rglob("train/annotations/xmls"))

if not img_dirs or not xml_dirs:
    print("ERROR: Could not find train/images or train/annotations/xmls")
    print("Directory structure:", list(raw_dir.rglob("*"))[:20])
else:
    img_dir = img_dirs[0]
    xml_dir = xml_dirs[0]

    # Collect all valid image-annotation pairs
    pairs = []
    for xml_file in sorted(xml_dir.glob("*.xml")):
        img_file = img_dir / xml_file.name.replace(".xml", ".jpg")
        if img_file.exists():
            pairs.append((img_file, xml_file))

    print(f"Found {len(pairs)} image-annotation pairs")

    # Shuffle and split: 85% train, 15% val
    random.seed(42)
    random.shuffle(pairs)
    split = int(0.85 * len(pairs))
    train_pairs = pairs[:split]
    val_pairs = pairs[split:]

    print(f"Train: {len(train_pairs)}, Val: {len(val_pairs)}")

    # Convert and copy
    for split_name, split_pairs in [("train", train_pairs), ("val", val_pairs)]:
        img_out = out_dir / "images" / split_name
        lbl_out = out_dir / "labels" / split_name
        img_out.mkdir(parents=True, exist_ok=True)
        lbl_out.mkdir(parents=True, exist_ok=True)

        skipped = 0
        for img_path, xml_path in split_pairs:
            img = PILImage.open(img_path)
            w, h = img.size

            yolo_lines = convert_voc_to_yolo(xml_path, w, h)
            if not yolo_lines:
                skipped += 1
                continue

            # Copy image
            shutil.copy(img_path, img_out / img_path.name)

            # Write YOLO label
            label_file = lbl_out / img_path.name.replace(".jpg", ".txt")
            label_file.write_text("\n".join(yolo_lines))

        print(f"{split_name}: {len(list(img_out.glob('*.jpg')))} images ({skipped} skipped — no valid labels)")

    print("\nConversion complete!")

### B2: Create dataset config and train

In [ ]:
import yaml

# Write the dataset YAML config
dataset_config = {
    "path": str(Path("datasets/rdd2022").resolve()),
    "train": "images/train",
    "val": "images/val",
    "names": {i: name for i, name in enumerate(CLASS_NAMES)},
}

config_path = Path("rdd2022.yaml")
config_path.write_text(yaml.dump(dataset_config, default_flow_style=False))
print("Dataset config written to rdd2022.yaml")
print(config_path.read_text())

In [ ]:
# Train YOLOv8n on RDD2022 (detection, not segmentation)
model_rdd = YOLO("yolov8n.pt")

results_rdd = model_rdd.train(
    data="rdd2022.yaml",
    epochs=80,
    imgsz=640,
    batch=16,
    name="pavescan_rdd2022",
    project="runs/detect",
    patience=15,
    save=True,
    plots=True,
)

### B3: Evaluate and download RDD2022 model

In [ ]:
# Evaluate
best_rdd = YOLO("runs/detect/pavescan_rdd2022/weights/best.pt")
metrics_rdd = best_rdd.val()

print("\n=== RDD2022 Model Performance ===")
print(f"mAP50:    {metrics_rdd.box.map50:.3f}")
print(f"mAP50-95: {metrics_rdd.box.map:.3f}")

# Per-class results
for i, name in enumerate(CLASS_NAMES):
    if i < len(metrics_rdd.box.ap50):
        print(f"  {name}: AP50 = {metrics_rdd.box.ap50[i]:.3f}")

In [ ]:
from google.colab import files
import shutil

# Download RDD2022 weights
best_weights = "runs/detect/pavescan_rdd2022/weights/best.pt"
output_name = "pavescan_rdd2022.pt"
shutil.copy(best_weights, output_name)

print(f"Downloading {output_name}...")
print("After download, copy this file to: pavescan-ai/models/pavescan_rdd2022.pt")
files.download(output_name)